# RAG-IDEArq — Indexación Mejorada

Indexación de PDFs arqueológicos + GeoJSON en Weaviate con:
- Chunking parametrizable (800/50 por defecto, editable en `src/config.py`)
- Metadata enriquecida (year, doi, authors, language, periodo, region)
- GeoJSON como documentos (1 feature = 1 doc, sin chunking)
- Una colección por modelo de embedding (PDFs + GeoJSON mezclados)
- Batching robusto con verificación post-index
- Checkpoint cada 50 docs (resume on crash)

**Sin chonkie** — mantiene `RecursiveCharacterTextSplitter` para comparar después.

In [1]:
# Cell 1: Setup
import os
import sys
import re
import json
import time
import pickle
import logging
import gc
import torch
import subprocess
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

# Add project root to path

try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
except NameError:
    # Jupyter notebook: usar el directorio de trabajo actual
    PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

from src.config import (
    CHUNK_SIZE, CHUNK_OVERLAP, SEPARATORS,
    MIN_CHUNK_LENGTH, MAX_CHUNK_LENGTH, MAX_DIGIT_RATIO,
    WEAVIATE_URL, EMBEDDINGS, collection_name, INDEX_PROPERTIES_FULL,
    INGESTA_DIR, GEOJSON_DIR, RESULTS_DIR,
)

import weaviate
from weaviate.classes.config import Configure, DataType, Property
from langchain_weaviate import WeaviateVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader, PDFPlumberLoader
from langchain_core.documents import Document
from langdetect import detect
from langchain_core.embeddings import Embeddings 
from sentence_transformers import SentenceTransformer

class E5InstructEmbeddings(Embeddings):
    """Custom embedding class for E5 Instruct models."""
    
    def __init__(self, model_name="intfloat/multilingual-e5-large-instruct", device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.model = SentenceTransformer(model_name, device=self.device)
    
    def embed_documents(self, texts):
        prefixed = [f"passage: {t}" for t in texts]
        return self.model.encode(prefixed, device=self.device).tolist()
    
    def embed_query(self, text):
        prefixed = f"query: {text}"
        return self.model.encode([prefixed], device=self.device)[0].tolist()

print(f"Project root: {PROJECT_ROOT}")
print(f"Chunking: {CHUNK_SIZE} chars, overlap {CHUNK_OVERLAP}")
print(f"Ingesta dir: {INGESTA_DIR}")
print(f"GeoJSON dir: {GEOJSON_DIR}")
print(f"Embeddings: {list(EMBEDDINGS.keys())}")

Project root: /home/raglinux/RAG
Chunking: 800 chars, overlap 50
Ingesta dir: /home/raglinux/RAG/data/ingesta
GeoJSON dir: /home/raglinux/RAG/data/geojson
Embeddings: ['all-MiniLM-L6-v2', 'gte-multilingual-base', 'e5-large-instruct']


In [2]:
# Cell 2: CUDA memory config + Weaviate connection
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def safe_empty_cache():
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        except Exception:
            pass
    gc.collect()

safe_empty_cache()

# Connect to Weaviate
w_client = weaviate.connect_to_local(
    host=WEAVIATE_URL.replace("http://", "").replace("https://", "").split(":")[0],
    port=8080,
    grpc_port=50051,
)
print(f"Weaviate connected: {w_client.is_ready()}")
print(f"Existing collections: {list(w_client.collections.list_all().keys())}")

Weaviate connected: True
Existing collections: ['BgeE5', 'BgeE5Chunk1000Sim05', 'IdearqGTE_800_50_v2', 'IdearqLinq', 'IdearqMiniLM_800_50_v2', 'IdearqQwen3', 'Idearqe5largeinstruct_800_50_v2']


/home/raglinux/env_rag/lib/python3.12/site-packages/weaviate/warnings.py:93: DeprecationWarning: Dep005: You are using weaviate-client version 4.17.0. The latest version is 4.22.0.
            Consider upgrading to the latest version. See https://weaviate.io/developers/weaviate/client-libraries/python for details.
  warnings.warn(


In [3]:
# Cell 3: Metadata extraction helpers
def extract_year(text: str) -> Optional[int]:
    match = re.search(r'\b(19|20)\d{2}\b', text[:3000])
    return int(match.group()) if match else None

def extract_doi(text: str) -> Optional[str]:
    match = re.search(r'10\.\d{4,9}/[-._;()/:A-Z0-9]+', text[:3000], re.IGNORECASE)
    return match.group() if match else None

def detect_language(text: str) -> str:
    try:
        return detect(text[:1000])
    except Exception:
        return "unknown"

PERIODOS = {
    "paleolitico": ["paleolítico", "paleolithic", "upper paleolithic"],
    "mesolitico": ["mesolítico", "mesolithic"],
    "neolitico": ["neolítico", "neolithic", "neolitización"],
    "calcolitico": ["calcolítico", "chalcolithic", "cobre", "edad del cobre"],
    "bronce": ["bronce", "bronze age", "bronce final"],
    "hierro": ["hierro", "iron age", "edad del hierro"],
}

REGIONES = [
    "andalucia", "andalucía", "extremadura", "castilla", "león", "leon",
    "portugal", "catalunya", "cataluña", "aragon", "aragón", "galicia",
    "asturias", "cantabria", "valencia", "murcia", "baleares", "balears",
    "navarra", "euskadi", "rioja", "mancha",
]

def extract_periodo(text: str) -> Optional[str]:
    lower = text[:3000].lower()
    for periodo, kws in PERIODOS.items():
        if any(k in lower for k in kws):
            return periodo
    return None

def extract_region(text: str) -> Optional[str]:
    lower = text[:3000].lower()
    for r in REGIONES:
        if r in lower:
            return r
    return None

def extract_authors_heuristic(text: str) -> str:
    lines = text[:2000].split('\n')
    for line in lines[:10]:
        line = line.strip()
        if 10 < len(line) < 100 and not line.startswith(('http', 'doi', '10.')):
            if re.match(r'^[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+(\s+[,y&]\s+[A-ZÁÉÍÓÚÑ][a-záéíóúñ]+)+', line):
                return line
    return ""

def extract_metadata(pdf_path: Path, page_text: str, fitz_doc) -> Dict[str, Any]:
    meta = fitz_doc.metadata if hasattr(fitz_doc, 'metadata') else {}
    doc_title = meta.get('title', '') or pdf_path.stem

    year = extract_year(page_text)
    doi = extract_doi(page_text)
    authors = extract_authors_heuristic(page_text)
    language = detect_language(page_text)
    periodo = extract_periodo(page_text)
    region = extract_region(page_text)

    return {
        'title': doc_title,
        'year': year,
        'doi': doi,
        'authors': authors,
        'language': language,
        'periodo': periodo,
        'region': region,
    }

In [4]:
# Cell 4: Chunk validation + PDF loading
def is_valid_chunk(text: str) -> bool:
    if len(text) < MIN_CHUNK_LENGTH:
        return False
    if len(text) > MAX_CHUNK_LENGTH:
        return False
    digit_ratio = sum(c.isdigit() for c in text) / max(len(text), 1)
    if digit_ratio > MAX_DIGIT_RATIO:
        return False
    return True

def load_pdfs(ingesta_dir: Path) -> Tuple[List[Document], List[str]]:
    pdf_files = sorted(ingesta_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDFs in {ingesta_dir}")

    all_docs = []
    failed = []

    for i, pdf_file in enumerate(pdf_files):
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            fitz_docs = loader.load()

            combined_content = "\n\n".join([d.page_content for d in fitz_docs])
            meta = extract_metadata(pdf_file, combined_content[:3000], fitz_docs[0] if fitz_docs else None)

            doc = Document(
                page_content=combined_content,
                metadata={
                    'source': str(pdf_file),
                    'filename': pdf_file.name,
                    'total_pages': len(fitz_docs),
                    'file_type': 'pdf',
                    'doc_type': 'pdf',
                    'doc_index': i,
                    **meta,
                }
            )
            all_docs.append(doc)

            if (i + 1) % 50 == 0:
                print(f"  Loaded {i + 1}/{len(pdf_files)} PDFs")

        except Exception as e:
            failed.append(pdf_file.name)
            logging.warning(f"Error loading '{pdf_file.name}': {e}")

    print(f"Loaded {len(all_docs)} PDFs, {len(failed)} failed")
    if failed:
        print(f"Failed files: {failed[:10]}{'...' if len(failed) > 10 else ''}")

    return all_docs, failed

In [ ]:
# Cell 5: Load GeoJSON as documents
def load_geojson(geojson_dir: Path) -> List[Document]:
    geo_files = sorted(geojson_dir.glob("*.geojson"))
    print(f"Found {len(geo_files)} GeoJSON files in {geojson_dir}")

    all_docs = []
    for geo_path in geo_files:
        try:
            with open(geo_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            features = data.get('features', [])
            for idx, feat in enumerate(features):
                geom = feat.get('geometry') or {}
                if geom.get('type') != 'Point':
                    continue
                coords = geom.get('coordinates') or []
                if len(coords) < 2:
                    continue

                props = feat.get('properties') or {}
                lon, lat = float(coords[0]), float(coords[1])

                # Build text representation
                text_parts = []
                nombre = props.get('yacimiento', props.get('name', 'Desconocido'))
                text_parts.append(f"Yacimiento: {nombre}")
                if props.get('unidad_territorial'):
                    text_parts.append(f"Ubicación: {props['unidad_territorial']}")
                if props.get('tipologia_crono'):
                    text_parts.append(f"Tipología: {props['tipologia_crono']}")
                if props.get('descripcion'):
                    desc = str(props['descripcion'])[:500]
                    text_parts.append(f"Descripción: {desc}")
                if props.get('dataciones_c_14'):
                    text_parts.append(f"Dataciones C14: {props['dataciones_c_14']}")

                doc = Document(
                    page_content="\n".join(text_parts),
                    metadata={
                        'source': str(geo_path),
                        'filename': geo_path.name,
                        'doc_type': 'yacimiento',
                        'doc_index': idx,
                        'lat': lat,
                        'lon': lon,
                        'yacimiento_id': props.get('yacimiento_id', props.get('id')),
                        'yacimiento_nombre': nombre,
                        'unidad_territorial': props.get('unidad_territorial', ''),
                        'tipologia_crono': props.get('tipologia_crono', ''),
                        'title': nombre,
                        'language': 'es',
                        'chunking_method': 'geojson_1doc_1feature',
                    }
                )
                all_docs.append(doc)

        except Exception as e:
            print(f"  Error loading {geo_path.name}: {e}")

    print(f"Loaded {len(all_docs)} yacimiento documents from {len(geo_files)} GeoJSON files")
    return all_docs

In [ ]:
# Cell 6: Chunk documents (PDFs only)
def chunk_documents(docs: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=SEPARATORS,
        add_start_index=True,
    )

    all_chunks = []
    total_raw = 0
    total_valid = 0

    for doc_idx, doc in enumerate(docs):
        raw_chunks = splitter.split_documents([doc])
        total_raw += len(raw_chunks)

        valid_chunks = []
        for i, chunk in enumerate(raw_chunks):
            if is_valid_chunk(chunk.page_content):
                chunk.metadata.update({
                    'chunk_index': i,
                    'total_chunks_from_doc': len(raw_chunks),
                    'chunking_method': f'recursive_{CHUNK_SIZE}_{CHUNK_OVERLAP}',
                })
                valid_chunks.append(chunk)

        total_valid += len(valid_chunks)
        all_chunks.extend(valid_chunks)

    print(f"Chunks: {total_raw} raw → {total_valid} valid ({100*total_valid/max(total_raw,1):.1f}%)")
    print(f"Average chunks per doc: {total_valid/max(len(docs),1):.1f}")

    return all_chunks

In [ ]:
# Cell 7: Create Weaviate collection with full schema
DATA_TYPE_MAP = {
    "TEXT": DataType.TEXT,
    "INT": DataType.INT,
    "NUMBER": DataType.NUMBER,
    "BOOL": DataType.BOOL,
}

def create_collection(name: str) -> None:
    if w_client.collections.exists(name):
        print(f"  Deleting existing collection: {name}")
        w_client.collections.delete(name)

    properties = [
        Property(name=pname, data_type=DATA_TYPE_MAP[ptype])
        for pname, ptype in INDEX_PROPERTIES_FULL
    ]

    w_client.collections.create(
        name=name,
        properties=properties,
        vector_index_config=Configure.VectorIndex.hnsw(),
        vectorizer_config=Configure.Vectorizer.none(),
    )
    print(f"  Created collection: {name} with {len(properties)} properties")

In [ ]:
# Cell 8: Safe embedding init with CPU fallback
def safe_init_embeddings(model_name: str, trust_remote_code: bool = False):
    try:
        emb = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": "cuda"},
            encode_kwargs={"device": "cuda"},
        )
        return emb, "cuda"
    except RuntimeError as e:
        print(f"  [OOM on CUDA] Falling back to CPU: {e}")
        safe_empty_cache()
        emb = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"device": "cpu"},
        )
        return emb, "cpu"
    except Exception as e:
        print(f"  [Error] Using CPU: {e}")
        emb = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={"device": "cpu"},
            encode_kwargs={"device": "cpu"},
        )
        return emb, "cpu"

In [ ]:
# Cell 9: Index documents for one embedding model
def index_for_embedding(emb_key: str, emb_cfg: dict, chunks: List[Document], checkpoint_path: Path) -> Dict[str, Any]:
    coll_name = collection_name(emb_key)
    print(f"\n{'='*60}")
    print(f"Embedding: {emb_key}")
    print(f"Collection: {coll_name}")
    print(f"Chunks to index: {len(chunks)}")
    print(f"{'='*60}")

    create_collection(coll_name)

    emb, device = safe_init_embeddings(
        emb_cfg["model_name"],
        emb_cfg.get("trust_remote_code", False)
    )
    print(f"  Model loaded on {device}: {emb_cfg['model_name']}")

    vs = WeaviateVectorStore(
        client=w_client,
        index_name=coll_name,
        text_key="content",
        embedding=emb,
        attributes=[p[0] for p in INDEX_PROPERTIES_FULL if p[0] != "content"],
    )

    batch_size = 10
    start_idx = 0

    if checkpoint_path.exists():
        with open(checkpoint_path, 'rb') as f:
            ckpt = pickle.load(f)
        start_idx = ckpt.get('indexed', 0)
        print(f"  Resuming from checkpoint: {start_idx}/{len(chunks)}")

    indexed = start_idx
    errors = 0
    t0 = time.time()

    for i in range(start_idx, len(chunks), batch_size):
        batch = chunks[i:i + batch_size]
        try:
            vs.add_documents(batch)
            indexed += len(batch)

            if indexed % 50 == 0:
                with open(checkpoint_path, 'wb') as f:
                    pickle.dump({'indexed': indexed, 'collection': coll_name}, f)
                elapsed = time.time() - t0
                rate = indexed / elapsed if elapsed > 0 else 0
                print(f"  [{indexed}/{len(chunks)}] {rate:.1f} chunks/s")

        except Exception as e:
            errors += len(batch)
            print(f"  Error at chunk {i}: {e}")
            safe_empty_cache()

    coll = w_client.collections.get(coll_name)
    agg = coll.aggregate.over_all(total_count=True)
    obj_count = agg.total_count

    elapsed = time.time() - t0
    print(f"\n  Results: {indexed} indexed, {errors} errors, {obj_count} objects in Weaviate")
    print(f"  Time: {elapsed:.1f}s ({indexed/elapsed:.1f} chunks/s)")

    if checkpoint_path.exists():
        checkpoint_path.unlink()

    return {
        "embedding": emb_key,
        "collection": coll_name,
        "indexed": indexed,
        "errors": errors,
        "object_count": obj_count,
        "time_s": elapsed,
        "device": device,
    }

In [ ]:
# Cell 10: Run full indexing pipeline
print("\n" + "="*60)
print("RAG-IDEArq — Indexación Mejorada")
print(f"Chunking: {CHUNK_SIZE}/{CHUNK_OVERLAP}")
print(f"Embeddings: {list(EMBEDDINGS.keys())}")
print("="*60 + "\n")

# 1. Load PDFs
print("\n[1/5] Loading PDFs...")
docs_pdf, failed = load_pdfs(Path(INGESTA_DIR))
if not docs_pdf:
    print("ERROR: No PDFs loaded. Check INGESTA_DIR.")
    sys.exit(1)

# 2. Load GeoJSON
print("\n[2/5] Loading GeoJSON...")
docs_geo = load_geojson(Path(GEOJSON_DIR))

# 3. Combine
all_docs = docs_pdf + docs_geo
print(f"\n[3/5] Total documents: {len(docs_pdf)} PDFs + {len(docs_geo)} yacimientos = {len(all_docs)}")

# 4. Chunk (PDFs only, GeoJSON already 1-doc-1-feature)
print("\n[4/5] Chunking PDF documents (GeoJSON already 1-doc-1-feature)...")
chunks_pdf = chunk_documents(docs_pdf)
chunks = chunks_pdf + docs_geo
print(f"Total chunks: {len(chunks_pdf)} (PDF) + {len(docs_geo)} (GeoJSON) = {len(chunks)}")

# 5. Index for each embedding
print("\n[5/5] Indexing into Weaviate...")
results = []
for emb_key, emb_cfg in EMBEDDINGS.items():
    ckpt_path = PROJECT_ROOT / f".index_ckpt_{collection_name(emb_key)}.pkl"
    result = index_for_embedding(emb_key, emb_cfg, chunks, ckpt_path)
    results.append(result)
    safe_empty_cache()

# 6. Report
print("\n[6/6] Final Report")
print("="*60)
for r in results:
    status = "OK" if r["errors"] == 0 and r["indexed"] == r["object_count"] else "WARN"
    n_geo = len(docs_geo)
    n_pdf = r["indexed"] - n_geo
    print(f"  [{status}] {r['embedding']} → {r['collection']}")
    print(f"         Indexed: {r['indexed']} ({n_pdf} PDF + {n_geo} GeoJSON), Errors: {r['errors']}")
    print(f"         Device: {r['device']}, Time: {r['time_s']:.1f}s")

print(f"\nFailed PDFs: {len(failed)}")
if failed:
    for f in failed[:5]:
        print(f"  - {f}")

print("\nDone! Collections ready for evaluation.")

### Colecciones indexadas

In [ ]:
import sys
from pathlib import Path

# Fix para Jupyter
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent.parent

sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

import weaviate
from src.config import WEAVIATE_URL, EMBEDDINGS, collection_name

# Conectar a Weaviate
host = WEAVIATE_URL.replace("http://", "").replace("https://", "").split(":")[0]
w_client = weaviate.connect_to_local(host=host, port=8080, grpc_port=50051)

print("="*60)
print("Colecciones en Weaviate")
print("="*60)
print(f"Server: {host}:8080")
print(f"Ready: {w_client.is_ready()}\n")

collections = w_client.collections.list_all()
if not collections:
    print("No hay colecciones indexadas.")
else:
    print(f"{'Colección':<40} {'Objetos':>10}")
    print("-"*60)
    for name in sorted(collections.keys()):
        coll = w_client.collections.get(name)
        agg = coll.aggregate.over_all(total_count=True)
        obj_count = agg.total_count or 0
        print(f"{name:<40} {obj_count:>10}")
    

w_client.close()

Colecciones en Weaviate
Server: localhost:8080
Ready: True

ColecciÃ³n                                  Objetos
------------------------------------------------------------
BgeE5                                             0
BgeE5Chunk1000Sim05                          215662
IdearqGTE_800_50_v2                               0
IdearqLinq                                     9035
IdearqMiniLM_800_50_v2                        53758
IdearqQwen3                                    9035

Colecciones esperadas (segÃºn config.py):
â IdearqMiniLM_800_50_v2
â IdearqGTE_800_50_v2
â Idearqe5largeinstruct_800_50_v2


/home/raglinux/env_rag/lib/python3.12/site-packages/weaviate/warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
/tmp/ipykernel_3473526/3258674393.py:35: ResourceWarning: unclosed <socket.socket fd=86, family=2, type=1, proto=6, laddr=('127.0.0.1', 44288), raddr=('127.0.0.1', 8080)>
  coll = w_client.collections.get(name)


## CARGA, CHUNKING E INDEXACIÓN DE PDFS DE UN SOLO MODELO
Borra y crea una colección nueva

In [ ]:

# Imports
import os
import sys
import re
import json
import time
import gc
import torch
from pathlib import Path
from typing import List, Dict, Any, Optional

# Setup paths
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

from src.config import (
    CHUNK_SIZE, CHUNK_OVERLAP, SEPARATORS,
    MIN_CHUNK_LENGTH, MAX_CHUNK_LENGTH, MAX_DIGIT_RATIO,
    WEAVIATE_URL, EMBEDDINGS, collection_name, INDEX_PROPERTIES_FULL,
    INGESTA_DIR,
)

import weaviate
from weaviate.classes.config import Configure, DataType, Property
from langchain_weaviate import WeaviateVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langdetect import detect
from langchain_core.embeddings import Embeddings
from sentence_transformers import SentenceTransformer


# Clase E5 
class E5InstructEmbeddings(Embeddings):
    """Custom embedding class for E5 Instruct models."""
    
    def __init__(self, model_name="intfloat/multilingual-e5-large-instruct", device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        self.model = SentenceTransformer(model_name, device=self.device)
    
    def embed_documents(self, texts):
        prefixed = [f"passage: {t}" for t in texts]
        return self.model.encode(prefixed, device=self.device).tolist()
    
    def embed_query(self, text):
        prefixed = f"query: {text}"
        return self.model.encode([prefixed], device=self.device)[0].tolist()


# Funciones de metadata 
def extract_year(text: str) -> Optional[int]:
    match = re.search(r'\b(19|20)\d{2}\b', text[:3000])
    return int(match.group()) if match else None

def extract_doi(text: str) -> Optional[str]:
    match = re.search(r'10\.\d{4,9}/[-._;()/:A-Z0-9]+', text[:3000], re.IGNORECASE)
    return match.group() if match else None

def detect_language(text: str) -> str:
    try:
        return detect(text[:1000])
    except Exception:
        return "unknown"

PERIODOS = {
    "paleolitico": ["paleolitico", "paleolithic", "upper paleolithic"],
    "mesolitico": ["mesolitico", "mesolithic"],
    "neolitico": ["neolitico", "neolithic", "neolitizaciÃ³n"],
    "calcolitico": ["calcoli­tico", "chalcolithic", "cobre", "edad del cobre"],
    "bronce": ["bronce", "bronze age", "bronce final"],
    "hierro": ["hierro", "iron age", "edad del hierro"],
}

REGIONES = [
    "andalucia", "andaluci­a", "extremadura", "castilla", "leon", "leon",
    "portugal", "catalunya", "cataluÃ±a", "aragon", "aragon", "galicia",
    "asturias", "cantabria", "valencia", "murcia", "baleares", "balears",
    "navarra", "euskadi", "rioja", "mancha",
]

def extract_periodo(text: str) -> Optional[str]:
    lower = text[:3000].lower()
    for periodo, kws in PERIODOS.items():
        if any(k in lower for k in kws):
            return periodo
    return None

def extract_region(text: str) -> Optional[str]:
    lower = text[:3000].lower()
    for r in REGIONES:
        if r in lower:
            return r
    return None

def extract_authors_heuristic(text: str) -> str:
    lines = text[:2000].split('\n')
    for line in lines[:10]:
        line = line.strip()
        if 10 < len(line) < 100 and not line.startswith(('http', 'doi', '10.')):
            if re.match(r'^[A-ZÃÃÃÃÃÃ][a-zÃ¡Ã©Ã­Ã³ÃºÃ±]+(\s+[,y&]\s+[A-ZÃÃÃÃÃÃ][a-zÃ¡Ã©Ã­Ã³ÃºÃ±]+)+', line):
                return line
    return ""

def extract_metadata(pdf_path: Path, page_text: str, fitz_doc) -> Dict[str, Any]:
    meta = fitz_doc.metadata if hasattr(fitz_doc, 'metadata') else {}
    doc_title = meta.get('title', '') or pdf_path.stem

    return {
        'title': doc_title,
        'year': extract_year(page_text),
        'doi': extract_doi(page_text),
        'authors': extract_authors_heuristic(page_text),
        'language': detect_language(page_text),
        'periodo': extract_periodo(page_text),
        'region': extract_region(page_text),
    }


# Carga de PDFs 
def load_pdfs(ingesta_dir: Path) -> List[Document]:
    """Load all PDFs from ingesta directory."""
    pdf_files = sorted(ingesta_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDFs in {ingesta_dir}")

    all_docs = []
    failed = []

    for i, pdf_file in enumerate(pdf_files):
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            fitz_docs = loader.load()

            combined_content = "\n\n".join([d.page_content for d in fitz_docs])
            meta = extract_metadata(pdf_file, combined_content[:3000], fitz_docs[0] if fitz_docs else None)

            doc = Document(
                page_content=combined_content,
                metadata={
                    'source': str(pdf_file),
                    'filename': pdf_file.name,
                    'total_pages': len(fitz_docs),
                    'file_type': 'pdf',
                    'doc_type': 'pdf',
                    'doc_index': i,
                    **meta,
                }
            )
            all_docs.append(doc)

            if (i + 1) % 50 == 0:
                print(f"  Loaded {i + 1}/{len(pdf_files)} PDFs")

        except Exception as e:
            failed.append(pdf_file.name)
            print(f"  Error loading {pdf_file.name}: {e}")

    print(f"Loaded {len(all_docs)} PDFs, {len(failed)} failed")
    return all_docs


# Chunking 
def is_valid_chunk(text: str) -> bool:
    if len(text) < MIN_CHUNK_LENGTH:
        return False
    if len(text) > MAX_CHUNK_LENGTH:
        return False
    digit_ratio = sum(c.isdigit() for c in text) / max(len(text), 1)
    if digit_ratio > MAX_DIGIT_RATIO:
        return False
    return True

def chunk_documents(docs: List[Document]) -> List[Document]:
    """Split documents into chunks with validation."""
    splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=SEPARATORS,
        add_start_index=True,
    )

    all_chunks = []
    total_raw = 0
    total_valid = 0

    for doc_idx, doc in enumerate(docs):
        raw_chunks = splitter.split_documents([doc])
        total_raw += len(raw_chunks)

        valid_chunks = []
        for i, chunk in enumerate(raw_chunks):
            if is_valid_chunk(chunk.page_content):
                chunk.metadata.update({
                    'chunk_index': i,
                    'total_chunks_from_doc': len(raw_chunks),
                    'chunking_method': f'recursive_{CHUNK_SIZE}_{CHUNK_OVERLAP}',
                })
                valid_chunks.append(chunk)

        total_valid += len(valid_chunks)
        all_chunks.extend(valid_chunks)

    print(f"Chunks: {total_raw} raw â {total_valid} valid ({100*total_valid/max(total_raw,1):.1f}%)")
    return all_chunks

#EJECUCIÓN PRINCIPAL

print("\n" + "="*60)
print("INDEXACIÓN E5 - SOLO PDFs")
print("="*60)

# 1. Conectar a Weaviate
print("\n[1/4] Conectando a Weaviate...")
w_client = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
print(f"â Weaviate connected: {w_client.is_ready()}")

# 2. Cargar PDFs
print("\n[2/4] Cargando PDFs...")
docs = load_pdfs(Path(INGESTA_DIR))
if not docs:
    print("ERROR: No PDFs loaded")
    sys.exit(1)

# 3. Chunking
print("\n[3/4] Generando chunks...")
chunks = chunk_documents(docs)
if not chunks:
    print("ERROR: No valid chunks")
    sys.exit(1)

# 4. Indexar E5
print("\n[4/4] Indexando E5...")

emb_key = "e5-large-instruct"
emb_cfg = EMBEDDINGS[emb_key]
coll_name = collection_name(emb_key)

print(f"\n{'='*60}")
print(f"Embedding: {emb_key}")
print(f"Collection: {coll_name}")
print(f"Chunks to index: {len(chunks)}")
print(f"{'='*60}")

# Borrar colecciÓN existente
if w_client.collections.exists(coll_name):
    print(f"  Deleting existing collection: {coll_name}")
    w_client.collections.delete(coll_name)

# Crear nueva colección
properties = [
    Property(name=pname, data_type=DataType[ptype])
    for pname, ptype in INDEX_PROPERTIES_FULL
]

w_client.collections.create(
    name=coll_name,
    properties=properties,
    vector_index_config=Configure.VectorIndex.hnsw(),
    vectorizer_config=Configure.Vectorizer.none(),
)
print(f"Collection created: {coll_name} with {len(properties)} properties")

# Cargar modelo E5
emb = E5InstructEmbeddings(model_name=emb_cfg["model_name"])
device = emb.device
print(f"Model loaded on {device}")

# Crear vector store
vs = WeaviateVectorStore(
    client=w_client,
    index_name=coll_name,
    text_key="content",
    embedding=emb,
    attributes=[p[0] for p in INDEX_PROPERTIES_FULL if p[0] != "content"],
)

# Indexar en batches
batch_size = 5
indexed = 0
errors = 0
t0 = time.time()

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    try:
        vs.add_documents(batch)
        indexed += len(batch)
        
        if indexed % 100 == 0:
            elapsed = time.time() - t0
            rate = indexed / elapsed if elapsed > 0 else 0
            print(f"  [{indexed}/{len(chunks)}] {rate:.1f} chunks/s")
            
    except Exception as e:
        errors += len(batch)
        print(f"  Error at chunk {i}: {e}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        # Fallback: batch_size=1
        print(f"  Retrying with batch_size=1...")
        for chunk in batch:
            try:
                vs.add_documents([chunk])
                indexed += 1
            except Exception as e2:
                print(f"    Failed: {e2}")

# Verificar
coll = w_client.collections.get(coll_name)
agg = coll.aggregate.over_all(total_count=True)
obj_count = agg.total_count or 0

elapsed = time.time() - t0
print(f"\nâ {emb_key} done!")
print(f"   Indexed: {indexed}, Objects: {obj_count}, Errors: {errors}")
print(f"   Time: {elapsed:.1f}s ({indexed/elapsed:.1f} chunks/s)")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

w_client.close()

print("\n" + "="*60)
print("INDEXACIÓN E5 COMPLETADA")
print("="*60)


INDEXACIÃN E5 - SOLO PDFs

[1/4] Conectando a Weaviate...
â Weaviate connected: True

[2/4] Cargando PDFs...
Found 525 PDFs in /home/raglinux/RAG/data/ingesta


/home/raglinux/env_rag/lib/python3.12/site-packages/weaviate/warnings.py:302: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
/tmp/ipykernel_3133648/2746903777.py:223: ResourceWarning: unclosed <socket.socket fd=93, family=2, type=1, proto=6, laddr=('127.0.0.1', 42114), raddr=('127.0.0.1', 8080)>
  w_client = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute


  Loaded 50/525 PDFs
  Loaded 100/525 PDFs
  Loaded 150/525 PDFs
  Loaded 200/525 PDFs
  Loaded 250/525 PDFs
  Loaded 300/525 PDFs
  Loaded 350/525 PDFs
  Loaded 400/525 PDFs
  Loaded 450/525 PDFs
  Loaded 500/525 PDFs
Loaded 525 PDFs, 0 failed

[3/4] Generando chunks...
Chunks: 54161 raw â 53758 valid (99.3%)

[4/4] Indexando E5...

Embedding: e5-large-instruct
Collection: Idearqe5largeinstruct_800_50_v2
Chunks to index: 53758
  Deleting existing collection: Idearqe5largeinstruct_800_50_v2


/home/raglinux/env_rag/lib/python3.12/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(
/home/raglinux/env_rag/lib/python3.12/site-packages/weaviate/warnings.py:206: DeprecationWarning: Dep025: You are using the `vector_index_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead defining `vector_index_config` as a sub-argument.
            
  warnings.warn(


â Collection created: Idearqe5largeinstruct_800_50_v2 with 23 properties
â Model loaded on cuda
  [100/53758] 4.1 chunks/s
  [200/53758] 4.3 chunks/s
  [300/53758] 4.3 chunks/s
  [400/53758] 4.4 chunks/s
  [500/53758] 4.4 chunks/s
  [600/53758] 4.4 chunks/s
  [700/53758] 4.4 chunks/s
  [800/53758] 4.4 chunks/s
  [900/53758] 4.4 chunks/s
  [1000/53758] 4.4 chunks/s
  [1100/53758] 4.4 chunks/s
  [1200/53758] 4.4 chunks/s
  [1300/53758] 4.4 chunks/s
  [1400/53758] 4.5 chunks/s
  [1500/53758] 4.5 chunks/s
  [1600/53758] 4.5 chunks/s
  [1700/53758] 4.5 chunks/s
  [1800/53758] 4.5 chunks/s
  [1900/53758] 4.5 chunks/s
  [2000/53758] 4.5 chunks/s
  [2100/53758] 4.5 chunks/s
  [2200/53758] 4.5 chunks/s
  [2300/53758] 4.5 chunks/s
  [2400/53758] 4.5 chunks/s
  [2500/53758] 4.5 chunks/s
  [2600/53758] 4.5 chunks/s
  [2700/53758] 4.5 chunks/s
  [2800/53758] 4.5 chunks/s
  [2900/53758] 4.5 chunks/s
  [3000/53758] 4.5 chunks/s
  [3100/53758] 4.5 chunks/s
  [3200/53758] 4.5 chunks/s
  [3300/53758

## Añadir nuevos PDFs a colecciones existentes

In [ ]:
print("="*60)
print("AÑADIR PDFs NUEVOS A COLECCIONES EXISTENTES")
print("="*60)

# 1. Verificar que las colecciones existen
collections_exist = True
existing_objects = {}
for emb_key, emb_cfg in EMBEDDINGS.items():
    coll_name = collection_name(emb_key)
    if not w_client.collections.exists(coll_name):
        print(f"Collection '{coll_name}' does NOT exist")
        collections_exist = False
    else:
        coll = w_client.collections.get(coll_name)
        agg = coll.aggregate.over_all(total_count=True)
        existing_objects[coll_name] = agg.total_count or 0
        print(f"{coll_name}: {existing_objects[coll_name]} objetos")

if not collections_exist:
    print("\nAlgunas colecciones no existen. Ejecuta primero las celdas 1-10.")
else:
    # 2. Obtener lista de archivos YA indexados
    print("\n[1/4] Verificando archivos ya indexados...")
    ref_coll = w_client.collections.get(collection_name("all-MiniLM-L6-v2")) # CAMBIAR NOMBRE DE COLECCIÓN
    response = ref_coll.query.fetch_objects(
        limit=100000,
        return_properties=["filename"]
    )
    existing_files = set()
    for obj in response.objects:
        if obj.properties.get("filename"):
            existing_files.add(obj.properties["filename"])
    print(f"{len(existing_files)} archivos añadidos ya indexados")
    
    # 3. Cargar PDFs y filtrar solo los NUEVOS
    print("\n[2/4] Cargando PDFs y filtrando nuevos...")
    all_docs = load_pdfs(Path(INGESTA_DIR))
    new_docs = [d for d in all_docs if d.metadata.get("filename") not in existing_files]
    
    print(f"  Total PDFs: {len(all_docs)}")
    print(f"  Ya indexados: {len(existing_files)}")
    print(f"  Nuevos a añadir: {len(new_docs)}")
    
    if not new_docs:
        print("\nNo hay PDFs nuevos para indexar. Todos ya están en las colecciones.")
    else:
        # 4. Chunkear solo los PDFs nuevos
        print(f"\n[3/4] Chunking {len(new_docs)} PDFs nuevos...")
        new_chunks = chunk_documents(new_docs)
        print(f"{len(new_chunks)} chunks generados")
        
        # 5. Indexar en cada colección
        print(f"\n[4/4] Añadiendo a colecciones existentes...")
        for emb_key, emb_cfg in EMBEDDINGS.items():
            coll_name = collection_name(emb_key)
            before = existing_objects[coll_name]
            
            print(f"\n{'='*60}")
            print(f"Embedding: {emb_key} and {coll_name}")
            print(f"Objects before: {before}")
            print(f"Chunks to add: {len(new_chunks)}")
            print(f"{'='*60}")
            
            # Cargar modelo de embedding
            if emb_cfg.get("model_class") == "E5InstructEmbeddings":
                emb = E5InstructEmbeddings(model_name=emb_cfg["model_name"])
            else:
                model_kwargs = {"device": "cuda"}
                if emb_cfg.get("trust_remote_code", False):
                    model_kwargs["trust_remote_code"] = True
                emb = HuggingFaceEmbeddings(
                    model_name=emb_cfg["model_name"],
                    model_kwargs=model_kwargs,
                    encode_kwargs={"device": "cuda"},
                )
            
            # Conectar a colección EXISTENTE
            vs = WeaviateVectorStore(
                client=w_client,
                index_name=coll_name,
                text_key="content",
                embedding=emb,
                attributes=[p[0] for p in INDEX_PROPERTIES_FULL if p[0] != "content"],
            )
            
            # AÃ±adir chunks (CON RETRY)
            batch_size = 5
            indexed = 0
            errors = 0
            t0 = time.time()
            
            for i in range(0, len(new_chunks), batch_size):
                batch = new_chunks[i:i + batch_size]
                try:
                    vs.add_documents(batch)
                    indexed += len(batch)
                    if indexed % 100 == 0:
                        elapsed = time.time() - t0
                        rate = indexed / elapsed if elapsed > 0 else 0
                        print(f"  [{indexed}/{len(new_chunks)}] {rate:.1f} chunks/s")
                except Exception as e:
                    errors += len(batch)
                    print(f"  Error at batch {i}: {e}")
                    safe_empty_cache()
                    # Retry one by one
                    for doc in batch:
                        try:
                            vs.add_documents([doc])
                            indexed += 1
                            errors -= 1
                        except Exception as e2:
                            print(f"    Failed: {e2}")
            
            # Verificar
            coll = w_client.collections.get(coll_name)
            after = coll.aggregate.over_all(total_count=True).total_count or 0
            elapsed = time.time() - t0
            
            print(f"\n  Results:")
            print(f"    Indexed: {indexed}, Errors: {errors}")
            print(f"    Time: {elapsed:.1f}s")
            print(f"    Objects: {before} {after} (+{after-before})")
            
            safe_empty_cache()
        
        print("\n" + "="*60)
        print("Terminado")
        print("="*60)

In [1]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Capability:', torch.cuda.get_device_capability(0))


PyTorch: 2.8.0+cu129
CUDA version: 12.9
CUDA available: True
Device: NVIDIA GeForce RTX 5090
Capability: (12, 0)
